In [1]:
import pandas as pd
from scipy.stats import uniform, gamma
from astro_prost.associate import associate_sample

print("pandas", pd.__version__) 

pandas 2.3.3


In [2]:
raw = pd.read_parquet("Result/snia_results.parquet")
print("rows:", len(raw))
print("missing tns_name:", raw["tns_name"].isna().sum())
print("missing ra / decl:", raw["ra"].isna().sum(), "/", raw["decl"].isna().sum())
print("duplicated tns_name (excluding missing):", raw["tns_name"].dropna().duplicated().sum())
print("duplicated diaObjectId:", raw["diaObjectId"].duplicated().sum())

rows: 127
missing tns_name: 0
missing ra / decl: 0 / 0
duplicated tns_name (excluding missing): 26
duplicated diaObjectId: 0


In [3]:
df = (
    raw.dropna(subset=["tns_name", "ra", "decl"])
       .sort_values("nDiaSources", ascending=False)
       .drop_duplicates(subset="tns_name", keep="first")
       .sort_index()
       .rename(columns={"tns_name": "name", "decl": "dec"})
       .reset_index(drop=True)
)

print(df.shape)    
# For instance, this is an example of one of the duplicated TNS entries. We keep the entry with the largest "nDiaSources"
display(df[df["name"] == "2021hiz"][["name", "diaObjectId", "nDiaSources"]]) 

(101, 10)
True


,name,diaObjectId,nDiaSources
8,2021hiz,170028527695429710,24


In [4]:
display(df)

,diaObjectId,name,z,catalogue_object_id,tns_prefix,type,ra,dec,absMag,nDiaSources
0,170019717277810735,2026ctw,0.150000,"10000,342494,11570",SN,SN Ia,151.740293,1.927064,NaN,754
1,170028486191218743,2026qrh,0.100000,"10000,81273,866",SN,SN Ia,64.185342,-48.943229,NaN,88
2,170028500610711681,2024xom,0.044167,CGCG036-024,SN,SN Ia,150.746125,2.336469,NaN,165
3,170028510695916335,2025mbp,0.046160,2MASXJ10062208+0109317,SN,SN Ia,151.592617,1.158385,NaN,273
4,170028526486945858,2026fgt,0.282000,115901861891057367,SN,SN Ia,186.188879,6.589125,NaN,538
...,...,...,...,...,...,...,...,...,...,...
96,170648615579549734,2026uxs,0.115000,"10000,290964,1056",SN,SN Ia,210.988573,-6.980285,NaN,3
97,170653057310261886,2025nou,0.040716,2MASXJ22041416-0159498,SN,SN Ia,331.058529,-1.997490,NaN,2
98,170657505023098990,2026tzm,0.180000,1237663784196047153,SN,SN Ia,356.991032,0.193536,-19.0089,2
99,170661828398940941,2025bbx,0.067000,14470795-3930390,SN,SN Ia,221.785403,-39.509297,NaN,3


In [5]:
transients_dict = df.set_index("name").to_dict(orient="index")

N_TEST = None
transient_catalog = df[["name", "ra", "dec"]].head(N_TEST)

In [6]:
catalogs = ["glade", "decals", "panstarrs"]

priors = {"offset": uniform(loc=0, scale=10)}
likes = {"offset": gamma(a=0.75)}

hosts = associate_sample(
    transient_catalog,
    priors=priors,
    likes=likes,
    catalogs=catalogs,
    name_col="name",
    coord_cols=("ra", "dec"),
    parallel=False,
    save=False,
)

print(hosts.columns.tolist())
display(hosts)

2026-10-14 14:52:42,137- INFO - Conditioning association on the following properties: ['offset']
2026-10-14 14:52:44,238- INFO - Loaded GLADE+ catalog.
2026-10-14 14:52:44,265- INFO - 

Associating 2021ita at RA, DEC = 186.495177, 7.235433 (redshift nan)
2026-10-14 14:52:44,279- WARNING - Association failed. Host is likely outside the search cone.
2026-10-14 14:52:46,724- INFO - Association successful!
2026-10-14 14:52:46,752- INFO - 
    
Properties of best host (in decals dr9) for 2021ita:
    objID: 9906624197431893
    R.A. (deg): 186.495065
    Dec. (deg): 7.235318
    Posterior: 9.9404e-01
    Redshift: 0.1510 ± 0.0188 (PHOT)
    Absolute Magnitude: -21.5 ± 0.9 (r)
    Transient Offset: 0.6 ± 0.1 arcsec
    Transient Offset Posterior: 9.9404e-01
2026-10-14 14:52:46,752- INFO - 
    
Properties of 2nd best host (in decals dr9) for 2021ita:
    Redshift: 0.6744 ± 0.1422 (PHOT)
    Absolute Magnitude: -20.4 ± 1.1 (r)
    Transient Offset: 296.7 ± 0.1 arcsec
    Transient Offset Post

['name', 'ra', 'dec', 'idx', 'best_cat', 'best_cat_release', 'query_time', 'smallcone_posterior', 'missedcat_posterior', 'extra_cat_cols', 'host_objID', 'host_name', 'host_total_posterior', 'host_ra', 'host_dec', 'host_redshift_mean', 'host_redshift_std', 'host_redshift_info', 'host_absmag_mean', 'host_absmag_std', 'host_absmag_info', 'host_offset_mean', 'host_offset_std', 'host_offset_info', 'host_offset_posterior', 'host_2_objID', 'host_2_name', 'host_2_total_posterior', 'host_2_ra', 'host_2_dec', 'host_2_redshift_mean', 'host_2_redshift_std', 'host_2_redshift_info', 'host_2_absmag_mean', 'host_2_absmag_std', 'host_2_absmag_info', 'host_2_offset_mean', 'host_2_offset_std', 'host_2_offset_info', 'host_2_offset_posterior', 'any_posterior', 'none_posterior']


,name,ra,dec,idx,best_cat,best_cat_release,query_time,smallcone_posterior,missedcat_posterior,extra_cat_cols,...,host_2_redshift_info,host_2_absmag_mean,host_2_absmag_std,host_2_absmag_info,host_2_offset_mean,host_2_offset_std,host_2_offset_info,host_2_offset_posterior,any_posterior,none_posterior
0,2021ita,186.495177,7.235433,0,decals,dr9,2.382735,1.324398e-14,1.324398e-14,{},...,PHOT,-20.448565,1.129981,r,296.704467,0.099996,,0.000000,1.0,3.973195e-14
1,2025kj,212.662570,-5.893035,1,decals,dr9,1.633271,3.165373e-15,3.165373e-15,{},...,PHOT,-19.230169,1.159517,r,296.167854,0.101407,,0.000000,1.0,9.496118e-15
2,2026mkj,298.238031,-14.706206,2,panstarrs,dr2,7.405325,6.035867e-15,6.035867e-15,{},...,,0.000000,0.000000,,294.226459,0.097482,,0.000000,1.0,1.810760e-14
3,2025voy,299.603567,-22.207169,3,panstarrs,dr2,5.833227,3.202878e-15,3.202878e-15,{},...,,0.000000,0.000000,,145.715037,0.102042,,0.000000,1.0,9.608635e-15
4,2026lap,242.956759,-12.334252,4,panstarrs,dr2,5.915561,1.454596e-15,1.454596e-15,{},...,,0.000000,0.000000,,226.832282,0.095167,,0.000000,1.0,4.363787e-15
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,2026qiz,325.579412,-12.795119,96,decals,dr9,1.363029,2.881905e-13,2.881905e-13,{},...,PHOT,-21.289291,1.124630,r,294.053966,0.098926,,0.000000,1.0,8.645716e-13
96,2025rku,332.375696,-19.186636,97,panstarrs,dr2,4.213988,2.633847e-15,2.633847e-15,{},...,,0.000000,0.000000,,221.087914,0.093787,,0.000000,1.0,7.901541e-15
97,2024lah,185.996538,7.725874,98,decals,dr9,2.566973,3.837877e-15,3.837877e-15,{},...,PHOT,-20.094529,1.223428,r,296.496552,0.098708,,0.000000,1.0,1.151363e-14
98,2025imw,310.178614,-20.435643,99,glade,latest,0.006879,5.975368e-14,5.975368e-14,{},...,SPEC,-20.959993,0.722500,B,229.065780,0.097705,,0.007434,1.0,1.792610e-13


In [7]:
matched = hosts.dropna(subset=["host_objID"])
print(f"{len(matched)} of {len(hosts)} transients got a host")

hosts.to_csv("hosts.csv", index=False)

100 of 100 transients got a host
